In [1]:

import os

folder_path = "../data/raw/resumes_word/"
files = os.listdir(folder_path)

print(f"Number of files found: {len(files)}")
print(files[:5])  # show first 5 filenames as a sample


Number of files found: 228
['Abiral_Pandey_Fullstack_Java.docx', 'Achyuth Resume_8.docx', 'Adelina_Erimia_PMP1.docx', 'Adhi Gopalam - SM.docx', 'AjayKumar.docx']


In [2]:
# Extract text from .docx files and store in a DataFrame
from docx import Document
import pandas as pd

data = []

for filename in files:
    if filename.lower().endswith(".docx"):
        filepath = os.path.join(folder_path, filename)
        doc = Document(filepath)
        full_text = "\n".join([para.text for para in doc.paragraphs])
        data.append({"filename": filename, "text": full_text})

df = pd.DataFrame(data)
print(f"Number of resumes processed: {len(df)}")
df.head()

Number of resumes processed: 228


,filename,text
0,Abiral_Pandey_Fullstack_Java.docx,Name: Abiral Pandey\nEmail: abiral.pandey88@gm...
1,Achyuth Resume_8.docx,Achyuth\n540-999-8048\nachyuth.java88@gmail.co...
2,Adelina_Erimia_PMP1.docx,"Adelina Erimia, PMP, Six Sigma Green Belt, SMC..."
3,Adhi Gopalam - SM.docx,Adhi Gopalam\nadhigopalam@gmail.com\n281-212-3...
4,AjayKumar.docx,Ajay Kumar (CSM)\t \t\t Email/Skype: a...


In [3]:
# Add a new column to the DataFrame that contains the length of the text in each resume
df["text_length"] = df["text"].str.len()

print(df["text_length"].describe())
print("\nFiles with very short text (possible extraction problems):")
print(df[df["text_length"] < 200][["filename", "text_length"]])

count      228.000000
mean     18294.346491
std       7505.320847
min          0.000000
25%      13568.250000
50%      17831.000000
75%      22844.000000
max      51681.000000
Name: text_length, dtype: float64

Files with very short text (possible extraction problems):
                                             filename  text_length
43                      employer_mounika details.docx            0
131  Raju Goduguchinta_Technical Program Manager.docx            0
150             Resume Vishal PM - MSIS, PMP-PMI.docx            0
172                                   SAURABH_PM.docx            2


In [4]:
# List all files in the folder
all_files_in_folder = os.listdir(folder_path)
docx_files = [f for f in all_files_in_folder if f.endswith(".docx")]
non_docx = [f for f in all_files_in_folder if not f.endswith(".docx")]

print(f"Total items in folder: {len(all_files_in_folder)}")
print(f".docx files: {len(docx_files)}")
print(f"Non-.docx items: {non_docx}")

Total items in folder: 228
.docx files: 224
Non-.docx items: ['Madhu_BA_AW.DOCX', 'Murali_Project Manager QA.DOCX', 'Raja Santhosam_PM Scrum Master.DOCX', 'TeresaNeetipudi IT BA.DOCX']


In [5]:
df_clean = df[df["text_length"] >= 200].copy()

print(f"Resumes kept: {len(df_clean)}")
print(f"Resumes dropped (too short): {len(df) - len(df_clean)}")

# Save to CSV for the next notebook step
df_clean.to_csv("../data/raw/cvs_from_word.csv", index=False)
print("Saved to data/raw/cvs_from_word.csv")

Resumes kept: 224
Resumes dropped (too short): 4
Saved to data/raw/cvs_from_word.csv


In [6]:
#clean one document to check if it works before running on all 224
import spacy

nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    doc = nlp(text.lower())
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space and token.is_alpha
    ]
    return " ".join(tokens)

# Test on the first resume only, to check it works before running on all 224
sample_cleaned = clean_text(df_clean["text"].iloc[0])
print(sample_cleaned[:500])

abiral pandey email phone current location woonsocket rhode island visa status citizen summary dynamic individual year software development experience design development deployment maintenance production support web base client server business application oop java technology exposure phase software development life agile rup waterfall design develop web ui screen angular js develop angularjs controller service filter directive module application knowledge etl tool like kettle pentaho microsoft s


In [7]:
# Now apply the cleaning function to all resumes in the cleaned DataFrame
df_clean["cleaned_text"] = df_clean["text"].apply(clean_text)
print("Done cleaning all resumes.")
df_clean[["filename", "cleaned_text"]].head()

Done cleaning all resumes.


,filename,cleaned_text
0,Abiral_Pandey_Fullstack_Java.docx,abiral pandey email phone current location woo...
1,Achyuth Resume_8.docx,achyuth objective year strong software experie...
2,Adelina_Erimia_PMP1.docx,adelina erimia pmp sigma green belt smc projec...
3,Adhi Gopalam - SM.docx,adhi gopalam certify scrum master business ana...
4,AjayKumar.docx,ajay kumar csm email skype mob professional su...


In [8]:
#save files
df_clean.to_csv("../data/processed/cvs_cleaned.csv", index=False)
print("Saved to data/processed/cvs_cleaned.csv")

Saved to data/processed/cvs_cleaned.csv


In [9]:
# Quick peek at a random sample of filenames to get a sense of role variety
import random
sample_files = random.sample(list(df_clean["filename"]), 15)
for f in sample_files:
    print(f)

Mehul.docx
Varun Kumar.docx
Resume - Sr PM.docx
Shiuli Mahmud.docx
Manikanta P.docx
jagadeesh k.docx
Priya_Sharma.docx
Deepika Chintalapati.docx
V Ashok A.docx
Chandler_BA.docx
ram krishna.docx
mohid_rj.docx
Sai Krishna BA.docx
Ravi Pattar- Sr. Agile Program manager.docx
Rajendra-PMP-CSM.docx


In [10]:
SKILLS_LIST = {
    # Technical / Full Stack Java
    "java": ["java", "j2ee", "core java", "spring", "spring boot", "spring mvc", "hibernate", "microservices", "multithreading"],
    "frontend": ["angular", "angularjs", "react", "reactjs", "javascript", "typescript", "html", "css", "bootstrap", "jquery", "node", "nodejs"],
    "database": ["sql", "mysql", "oracle", "postgresql", "mongodb", "nosql", "pl sql", "dynamodb"],
    "cloud_devops": ["aws", "azure", "gcp", "docker", "kubernetes", "jenkins", "ci cd", "git", "github", "terraform", "devops"],
    "api": ["rest", "restful", "soap", "api", "web service", "webservices", "graphql", "postman"],
    "testing": ["junit", "selenium", "mockito", "unit testing", "automation testing", "test driven development", "tdd"],

    # Business Analysis
    "ba_tools": ["jira", "confluence", "visio", "sql", "excel", "power bi", "tableau", "sharepoint", "balsamiq"],
    "ba_skills": ["requirement gathering", "business requirement", "functional requirement", "gap analysis",
                  "use case", "user story", "stakeholder", "uat", "wireframe", "process improvement",
                  "business process", "data analysis", "root cause analysis", "swot"],

    # Project Management
    "pm_methodology": ["agile", "scrum", "waterfall", "kanban", "sprint", "rup", "lean", "scaled agile", "safe agile"],
    "pm_certification": ["pmp", "csm", "cspo", "safe", "six sigma", "prince2", "itil", "capm"],
    "pm_tools": ["ms project", "jira", "asana", "trello", "confluence", "smartsheet"],
    "pm_skills": ["risk management", "budget management", "resource allocation", "project planning",
                  "project lifecycle", "vendor management", "cross functional", "roadmap"],

    # Agile-specific 
    "agile_roles": ["scrum master", "product owner", "agile coach", "release train engineer"],
    "agile_ceremonies": ["daily standup", "sprint planning", "sprint review", "retrospective",
                          "backlog grooming", "backlog refinement", "sprint retrospective"],
    "agile_concepts": ["user story", "story point", "velocity", "burndown chart", "epic",
                        "product backlog", "definition of done", "acceptance criteria",
                        "minimum viable product", "mvp", "scrum of scrums"],

    # Education / seniority signals recruiters scan for
    "education": ["bachelor", "master", "mba", "computer science", "information technology", "engineering degree"],
    "seniority": ["senior", "lead", "junior", "manager", "director", "years of experience", "years experience"],

    # Work authorization 
    "work_auth": ["us citizen", "green card", "h1b", "opt", "cpt", "visa status", "work authorization"],

    # Domain knowledge 
    "domain": ["healthcare", "finance", "banking", "insurance", "retail", "telecom", "supply chain", "ecommerce"],

    # Soft skills  ATS-scanned
    "soft_skills": ["leadership", "communication", "stakeholder management", "team management",
                     "problem solving", "negotiation", "collaboration", "time management",
                     "critical thinking", "adaptability", "mentoring", "presentation skill"]
}

# Flatten into one list for easy searching
all_skills = [skill for category in SKILLS_LIST.values() for skill in category]
print(f"Total number of skill keywords: {len(all_skills)}")
print(all_skills)

Total number of skill keywords: 171
['java', 'j2ee', 'core java', 'spring', 'spring boot', 'spring mvc', 'hibernate', 'microservices', 'multithreading', 'angular', 'angularjs', 'react', 'reactjs', 'javascript', 'typescript', 'html', 'css', 'bootstrap', 'jquery', 'node', 'nodejs', 'sql', 'mysql', 'oracle', 'postgresql', 'mongodb', 'nosql', 'pl sql', 'dynamodb', 'aws', 'azure', 'gcp', 'docker', 'kubernetes', 'jenkins', 'ci cd', 'git', 'github', 'terraform', 'devops', 'rest', 'restful', 'soap', 'api', 'web service', 'webservices', 'graphql', 'postman', 'junit', 'selenium', 'mockito', 'unit testing', 'automation testing', 'test driven development', 'tdd', 'jira', 'confluence', 'visio', 'sql', 'excel', 'power bi', 'tableau', 'sharepoint', 'balsamiq', 'requirement gathering', 'business requirement', 'functional requirement', 'gap analysis', 'use case', 'user story', 'stakeholder', 'uat', 'wireframe', 'process improvement', 'business process', 'data analysis', 'root cause analysis', 'swot', '

In [11]:
# extracting skills from Abial pandey based on the skills list
def extract_skills(text, skills_list=all_skills):
    found = []
    for skill in skills_list:
        
        # remove simple stopwords like "of" so multi-word phrases still match
        skill_normalized = skill.replace(" of ", " ").replace(" the ", " ")
        if skill_normalized in text:
            found.append(skill)
    return found

# Test on the first resume
sample_skills = extract_skills(df_clean["cleaned_text"].iloc[0])
print(f"Skills found in {df_clean['filename'].iloc[0]}:")
print(sample_skills)

Skills found in Abiral_Pandey_Fullstack_Java.docx:
['java', 'core java', 'spring', 'spring mvc', 'hibernate', 'multithreading', 'angular', 'angularjs', 'javascript', 'html', 'css', 'bootstrap', 'jquery', 'node', 'sql', 'mysql', 'oracle', 'postgresql', 'mongodb', 'nosql', 'pl sql', 'git', 'github', 'rest', 'restful', 'soap', 'api', 'web service', 'junit', 'mockito', 'unit testing', 'jira', 'visio', 'sql', 'excel', 'use case', 'business process', 'agile', 'waterfall', 'rup', 'jira', 'bachelor', 'computer science', 'junior', 'us citizen', 'visa status', 'insurance', 'collaboration']


In [12]:
# apply the skill extraction function to all resumes in the cleaned DataFrame
df_clean["extracted_skills"] = df_clean["cleaned_text"].apply(extract_skills)
df_clean["num_skills_found"] = df_clean["extracted_skills"].apply(len)

print("Done extracting skills for all resumes.")
print(f"\nAverage number of skills found per resume: {df_clean['num_skills_found'].mean():.1f}")
print(f"Min: {df_clean['num_skills_found'].min()}, Max: {df_clean['num_skills_found'].max()}")

df_clean[["filename", "num_skills_found", "extracted_skills"]].head()

Done extracting skills for all resumes.

Average number of skills found per resume: 47.3
Min: 13, Max: 78


,filename,num_skills_found,extracted_skills
0,Abiral_Pandey_Fullstack_Java.docx,48,"[java, core java, spring, spring mvc, hibernat..."
1,Achyuth Resume_8.docx,50,"[java, core java, spring, spring boot, spring ..."
2,Adelina_Erimia_PMP1.docx,25,"[visio, excel, sharepoint, stakeholder, uat, p..."
3,Adhi Gopalam - SM.docx,57,"[java, oracle, soap, web service, automation t..."
4,AjayKumar.docx,78,"[java, sql, oracle, pl sql, git, github, rest,..."


In [13]:
df_clean.to_csv("../data/processed/cvs_with_skills.csv", index=False)
print("Saved to data/processed/cvs_with_skills.csv")

Saved to data/processed/cvs_with_skills.csv


In [14]:
import re

def extract_years_experience(text):
    # Look for patterns like: "8 year experience", "5 year", "10 yrs", "3+ year"
    pattern = r'(\d{1,2})\s*\+?\s*(?:year|yr)s?\b'
    matches = re.findall(pattern, text)
    
    if matches:
        # Convert all found numbers to integers, return the largest
        # (resumes often mention years per skill too, e.g. "5 year java", so we take the max as overall seniority signal)
        numbers = [int(m) for m in matches]
        return max(numbers)
    else:
        return None  # no match found

# Test on the first resume
sample_years = extract_years_experience(df_clean["cleaned_text"].iloc[0])
print(f"Years of experience found for {df_clean['filename'].iloc[0]}: {sample_years}")

Years of experience found for Abiral_Pandey_Fullstack_Java.docx: None


In [15]:
import re

def extract_years_experience(text):
    pattern = r'(\d{1,2})\s*\+?\s*(?:year|yr)s?\b'
    matches = re.findall(pattern, text.lower())
    
    if matches:
        numbers = [int(m) for m in matches]
        return max(numbers)
    else:
        return None

# Test on the first resume — using the ORIGINAL text, not cleaned_text
sample_years = extract_years_experience(df_clean["text"].iloc[0])
print(f"Years of experience found for {df_clean['filename'].iloc[0]}: {sample_years}")

Years of experience found for Abiral_Pandey_Fullstack_Java.docx: 6


In [16]:
df_clean["years_experience"] = df_clean["text"].apply(extract_years_experience)

print(f"Resumes with years detected: {df_clean['years_experience'].notna().sum()} out of {len(df_clean)}")
print(f"Resumes with NO years detected: {df_clean['years_experience'].isna().sum()}")
print(f"\nAverage years (where found): {df_clean['years_experience'].mean():.1f}")
print(f"Min: {df_clean['years_experience'].min()}, Max: {df_clean['years_experience'].max()}")

df_clean[["filename", "years_experience"]].head(10)

Resumes with years detected: 207 out of 224
Resumes with NO years detected: 17

Average years (where found): 10.2
Min: 4.0, Max: 67.0


,filename,years_experience
0,Abiral_Pandey_Fullstack_Java.docx,6.0
1,Achyuth Resume_8.docx,8.0
2,Adelina_Erimia_PMP1.docx,10.0
3,Adhi Gopalam - SM.docx,12.0
4,AjayKumar.docx,14.0
5,Akhil.profile.docx,8.0
6,Akhil_Sr BSA.docx,8.0
7,Alekhya Resume.docx,8.0
8,Amar Sr BSA.docx,8.0
9,Ami Jape.docx,10.0


In [17]:
# Find the resume(s) with suspiciously high years (likely false positives)
suspicious = df_clean[df_clean["years_experience"] > 40][["filename", "years_experience"]]
print("Suspicious high values:")
print(suspicious)

# Show a few resumes where nothing was detected
no_match = df_clean[df_clean["years_experience"].isna()][["filename"]].head(5)
print("\nSample resumes with no years detected:")
print(no_match)

Suspicious high values:
             filename  years_experience
13   AnilAgarwal.docx              67.0
19     avinash G.docx              50.0
158        sai k.docx              50.0

Sample resumes with no years detected:
                         filename
18   Avathika BA-Healthcare_.docx
42          Drakshajavauidev.docx
60                    jeck P.docx
61        Jennifer  M. Conte.docx
75  KY BA PM UPDATED RESUME .docx


In [18]:
# Look at the actual matched patterns in the suspicious resumes
sample_text = df_clean[df_clean["filename"] == "AnilAgarwal.docx"]["text"].iloc[0].lower()
pattern = r'\d{1,2}\s*\+?\s*(?:year|yr)s?\b'

for match in re.finditer(pattern, sample_text):
    start = max(0, match.start() - 30)
    end = min(len(sample_text), match.end() + 20)
    print(repr(sample_text[start:end]))
    print("---")

'\nsummary\n13 years of total profession'
---
'ect management methodologies \n6 years of rich onsite it e'
---
' decide with confidence® for 167 years. d&b’s global comme'
---


In [19]:
def extract_years_experience(text):
    pattern = r'(\d{1,2})\s*\+?\s*(?:year|yr)s?\b'
    matches = re.findall(pattern, text.lower())
    
    if matches:
        numbers = [int(m) for m in matches if int(m) <= 45]  # filter unrealistic values
        if numbers:
            return max(numbers)
    return None

# Re-run on all resumes
df_clean["years_experience"] = df_clean["text"].apply(extract_years_experience)

print(f"Resumes with years detected: {df_clean['years_experience'].notna().sum()} out of {len(df_clean)}")
print(f"Average years (where found): {df_clean['years_experience'].mean():.1f}")
print(f"Min: {df_clean['years_experience'].min()}, Max: {df_clean['years_experience'].max()}")

Resumes with years detected: 207 out of 224
Average years (where found): 9.6
Min: 4.0, Max: 25.0


In [20]:
df_clean.to_csv("../data/processed/cvs_with_skills.csv", index=False)
print("Saved to data/processed/cvs_with_skills.csv")

Saved to data/processed/cvs_with_skills.csv


In [21]:
def extract_education(text):
    text = text.lower()
    
    # Ordered from highest to lowest — we return the highest one found
    if any(kw in text for kw in ["phd", "doctorate", "ph.d"]):
        return "PhD"
    elif any(kw in text for kw in ["mba", "master of business administration"]):
        return "MBA"
    elif any(kw in text for kw in ["master", "m.s.", "ms degree", "msc", "m.tech"]):
        return "Master"
    elif any(kw in text for kw in ["bachelor", "b.s.", "bs degree", "bsc", "b.tech", "b.e."]):
        return "Bachelor"
    elif any(kw in text for kw in ["associate degree", "a.s.", "diploma"]):
        return "Associate"
    else:
        return "Not specified"

# Test on the first resume
sample_education = extract_education(df_clean["text"].iloc[0])
print(f"Education found for {df_clean['filename'].iloc[0]}: {sample_education}")

Education found for Abiral_Pandey_Fullstack_Java.docx: Bachelor


In [22]:
df_clean["education"] = df_clean["text"].apply(extract_education)

print("Education level distribution:")
print(df_clean["education"].value_counts())

Education level distribution:
education
Master           71
MBA              58
Bachelor         57
Not specified    35
Associate         2
PhD               1
Name: count, dtype: int64


In [23]:
df_clean.to_csv("../data/processed/cvs_with_skills.csv", index=False)
print("Saved to data/processed/cvs_with_skills.csv")

Saved to data/processed/cvs_with_skills.csv


In [24]:
import pandas as pd
import spacy

df_clean = pd.read_csv("../data/processed/cvs_with_skills.csv")
nlp = spacy.load("en_core_web_sm")

print(len(df_clean), "resumes loaded")
print("Has text column:", "text" in df_clean.columns)

224 resumes loaded
Has text column: True


In [25]:
# Step: Extract candidate location from resume text using spaCy

def extract_location(text):
    if pd.isna(text):
        return "Unknown"
    doc = nlp(str(text)[:300])  # scan first 300 chars - location is usually near the top
    for ent in doc.ents:
        if ent.label_ == "GPE":  # GPE = cities, states, countries
            return ent.text
    return "Unknown"

# test on first 10 resumes
for i in range(10):
    print(df_clean["filename"].iloc[i], "->", extract_location(df_clean["text"].iloc[i]))

Abiral_Pandey_Fullstack_Java.docx -> Rhode Island
Achyuth Resume_8.docx -> Client
Adelina_Erimia_PMP1.docx -> Unknown
Adhi Gopalam - SM.docx -> Unknown
AjayKumar.docx -> Unknown
Akhil.profile.docx -> Unknown
Akhil_Sr BSA.docx -> Unknown
Alekhya Resume.docx -> Unknown
Amar Sr BSA.docx -> Unknown
Ami Jape.docx -> ST


In [26]:
# Step: Generate synthetic location and university data for candidates (demo/placeholder data)

import random

us_states = ["California", "Texas", "New York", "Florida", "Illinois", "New Jersey",
             "Virginia", "Georgia", "North Carolina", "Pennsylvania", "Ohio", "Michigan",
             "Massachusetts", "Washington", "Arizona", "Rhode Island", "Wisconsin", "Kansas"]

universities = ["University of Texas", "New York University", "Stanford University",
                "Arizona State University", "University of Illinois", "Georgia Tech",
                "Penn State University", "Ohio State University", "University of Michigan",
                "Boston University", "University of Washington", "Rutgers University",
                "Virginia Tech", "University of Florida", "Michigan State University"]

random.seed(42)  # makes the random results reproducible (same every time you run it)

df_clean["location"] = [random.choice(us_states) for _ in range(len(df_clean))]
df_clean["university"] = [random.choice(universities) for _ in range(len(df_clean))]

print(df_clean[["filename", "location", "university"]].head(10))

                            filename        location  \
0  Abiral_Pandey_Fullstack_Java.docx         Florida   
1              Achyuth Resume_8.docx      California   
2           Adelina_Erimia_PMP1.docx  North Carolina   
3             Adhi Gopalam - SM.docx         Georgia   
4                     AjayKumar.docx         Georgia   
5                 Akhil.profile.docx        Illinois   
6                  Akhil_Sr BSA.docx         Florida   
7                Alekhya Resume.docx          Kansas   
8                   Amar Sr BSA.docx        New York   
9                      Ami Jape.docx      Washington   

                  university  
0   Arizona State University  
1      University of Florida  
2  Michigan State University  
3        New York University  
4      Ohio State University  
5              Virginia Tech  
6      University of Florida  
7      University of Florida  
8     University of Michigan  
9        New York University  


In [27]:
# Step: Save updated CV data (with location + university) to CSV for Power BI

df_clean.to_csv("../data/processed/cvs_with_skills.csv", index=False)
print("Saved. New columns added: location, university")

Saved. New columns added: location, university
